# Prioritized Variant List

This notebook produces a prioritized variant table by merging the cardiovascular association results with gnomAD variant frequency data.

In [19]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

In [20]:
associations_path = Path("../data/interim/association_results.csv")
gnomad_path       = Path("../data/interim/gnomad_frequencies.csv")

out_dir       = Path("../data/final/")
out_excel_dir = Path("../data/final/excel/")

out_dir.mkdir(parents=True, exist_ok=True)
out_excel_dir.mkdir(parents=True, exist_ok=True)

out_csv              = out_dir       / "prioritized_variants.csv"
out_hf_csv           = out_dir       / "hf_prioritized_variants.csv"
out_top_variants_csv = out_dir       / "top_variants_per_gene.csv"

out_xlsx              = out_excel_dir / "prioritized_variants.xlsx"
out_hf_xlsx           = out_excel_dir / "hf_prioritized_variants.xlsx"
out_top_variants_xlsx = out_excel_dir / "top_variants_per_gene.xlsx"

In [21]:
associations_df = pd.read_csv(associations_path, sep=";", low_memory=False)
gnomad_df = pd.read_csv(gnomad_path, sep=";", low_memory=False)

print(f"Association results loaded: {associations_df.shape[0]:,} rows")
print(f"gnomAD frequencies loaded:  {gnomad_df.shape[0]:,} rows")

Association results loaded: 238,651 rows
gnomAD frequencies loaded:  2,917,852 rows


In [22]:
gnomad_merge = gnomad_df[[
    "rsID",
    "gene_symbol",
    "official_gene_symbol",
    "Functional_consequence",
    "MAX_freq",
    "MAX_freq_source",
    "MAX_freq_population",
    "Priority",
]].copy()

gnomad_merge = gnomad_merge.rename(columns={
    "official_gene_symbol": "gnomad_gene_symbol",
})

gnomad_merge = gnomad_merge.dropna(subset=["rsID"]).drop_duplicates(subset=["rsID"])

In [23]:
assoc_merge = associations_df[[
    "official_gene_symbol",
    "trait_name",
    "source_dataset",
    "rsID",
    "variant_id",
    "p_value",
    "effect_size",
    "odds_ratio",
    "effect_allele",
    "other_allele",
    "allele_frequency",
    "location_relative_to_gene",
    "distance_from_gene",
    "region_assembly",
    "chromosome",
    "position",
]].copy()

assoc_merge = assoc_merge.dropna(subset=["rsID", "p_value"])

print(f"Association records with rsID and p-value available for merge: {len(assoc_merge):,}")

Association records with rsID and p-value available for merge: 233,417


In [24]:
merged_df = assoc_merge.merge(
    gnomad_merge,
    on="rsID",
    how="inner",
)

merged_df = merged_df.drop(columns=["gnomad_gene_symbol"])

print(f"Variants with both association and gnomAD data: {len(merged_df):,} rows")
print(f"Unique rsIDs in merged table: {merged_df['rsID'].nunique():,}")

Variants with both association and gnomAD data: 225,358 rows
Unique rsIDs in merged table: 49,050


In [27]:
final_columns = [
    "official_gene_symbol",
    "trait_name",
    "source_dataset",
    "rsID",
    "variant_id",
    "p_value",
    "effect_size",
    "odds_ratio",
    "effect_allele",
    "other_allele",
    "allele_frequency",
    "location_relative_to_gene",
    "distance_from_gene",
    "region_assembly",
    "chromosome",
    "position",
    "Functional_consequence",
    "MAX_freq",
    "MAX_freq_source",
    "MAX_freq_population",
    "Priority",
]

prioritized_df = merged_df[final_columns].copy()

prioritized_df = prioritized_df.rename(columns={
    "official_gene_symbol": "gene",
    "trait_name": "phenotype",
    "Functional_consequence": "functional_consequence",
    "MAX_freq_source": "gnomad_max_freq_source",
    "MAX_freq_population": "gnomad_max_freq_population",
    "MAX_freq": "gnomad_max_freq",
    "Priority": "gnomad_priority",
})

prioritized_df = prioritized_df.sort_values(
    ["gene", "p_value"],
    ascending=[True, True],
    na_position="last"
).reset_index(drop=True)

print(f"Final prioritized variant table: {prioritized_df.shape[0]:,} rows, {prioritized_df.shape[1]} columns")
print(f"Unique rsIDs: {prioritized_df['rsID'].nunique():,}")
prioritized_df.head(10)

Final prioritized variant table: 225,358 rows, 21 columns
Unique rsIDs: 49,050


,gene,phenotype,source_dataset,rsID,variant_id,p_value,effect_size,odds_ratio,effect_allele,other_allele,allele_frequency,location_relative_to_gene,distance_from_gene,region_assembly,chromosome,position,functional_consequence,gnomad_max_freq,gnomad_max_freq_source,gnomad_max_freq_population,gnomad_priority
0,APOE,Alzheimer disease; age at onset,GWAS Catalog,rs429358,NaN,0.0,NaN,NaN,?,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missense_variant,0.226612,exome,afr,Primary
1,APOE,Cholesterol:total lipids ratio; high density l...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919344,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
2,APOE,Cholesterol:total lipids ratio; high density l...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919396,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
3,APOE,Cholesteryl esters:total lipids ratio; Cholest...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,T,NaN,0.074500,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
4,APOE,Cholesteryl esters:total lipids ratio; blood V...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919344,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
5,APOE,Cholesteryl esters:total lipids ratio; blood V...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919344,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
6,APOE,Cholesteryl esters:total lipids ratio; blood V...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,T,NaN,0.075100,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
7,APOE,Cholesteryl esters:total lipids ratio; blood V...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919797,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
8,APOE,Cholesteryl esters:total lipids ratio; high de...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919396,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
9,APOE,Cholesteryl esters:total lipids ratio; high de...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919344,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary


In [8]:
prioritized_df.to_excel(out_xlsx, index=False)
prioritized_df.to_csv(out_csv, sep=";", index=False)

## Top Variants Per Gene

In [17]:
all_genes = sorted(prioritized_df["gene"].unique())

top_variant_rows = []

for gene in all_genes:
    gene_df = prioritized_df[
        prioritized_df["gene"] == gene
    ].dropna(subset=["p_value", "rsID"])

    gene_df = gene_df.sort_values("p_value", ascending=True).drop_duplicates(subset=["rsID"])

    # Tier 1: genome-wide significant
    gws = gene_df[gene_df["p_value"] < 5e-8].copy()
    gws["significance_tier"] = "Genome-wide significant"

    # Tier 2: suggestive (not already in Tier 1)
    suggestive = gene_df[
        (gene_df["p_value"] >= 5e-8) & (gene_df["p_value"] < 1e-5)
    ].copy()
    suggestive["significance_tier"] = "Suggestive"

    # Tier 3: exploratory — only if gene has no variants in Tier 1 or Tier 2
    if len(gws) == 0 and len(suggestive) == 0:
        exploratory = gene_df.head(1).copy()
        exploratory["significance_tier"] = "Exploratory"
    else:
        exploratory = pd.DataFrame()

    top_variant_rows.append(pd.concat([gws, suggestive, exploratory], ignore_index=True))

top_variants_df = pd.concat(top_variant_rows, ignore_index=True)

print(f"Top variants per gene: {len(top_variants_df):,} records across {top_variants_df['gene'].nunique():,} genes")
print(f"Genome-wide significant: {(top_variants_df['significance_tier'] == 'Genome-wide significant').sum():,}")
print(f"Suggestive: {(top_variants_df['significance_tier'] == 'Suggestive').sum():,}")
print(f"Exploratory: {(top_variants_df['significance_tier'] == 'Exploratory').sum():,}")

Top variants per gene: 1,819 records across 51 genes
Genome-wide significant: 1,507
Suggestive: 311
Exploratory: 1


In [18]:
top_variants_df.to_csv(out_top_variants_csv, sep=";", index=False)
top_variants_df.to_excel(out_top_variants_xlsx, index=False)

## Heart Failure Prioritized Variants

In [28]:
hf_df = prioritized_df.copy()
hf_df["phenotype"] = hf_df["phenotype"].str.strip()
hf_df["phenotype"] = hf_df["phenotype"].replace("heart failure", "Heart failure")

hf_df = hf_df[hf_df["phenotype"] == "Heart failure"].copy().reset_index(drop=True)

In [29]:
hf_df["genome_build"] = hf_df["region_assembly"]

In [30]:
hf_columns = [
    "gene",
    "phenotype",
    "source_dataset",
    "rsID",
    "variant_id",
    "chromosome",
    "position",
    "genome_build",
    "effect_allele",
    "other_allele",
    "effect_size",
    "odds_ratio",
    "p_value",
    "allele_frequency",
    "location_relative_to_gene",
    "distance_from_gene",
    "functional_consequence",
    "gnomad_max_freq",
    "gnomad_max_freq_source",
    "gnomad_max_freq_population",
    "gnomad_priority",
]

hf_df = hf_df[hf_columns].reset_index(drop=True)

print(f"HF prioritized variant table: {len(hf_df):,} rows, {hf_df.shape[1]} columns")
print(f"Unique rsIDs: {hf_df['rsID'].nunique():,}")

HF prioritized variant table: 70,856 rows, 21 columns
Unique rsIDs: 48,390


In [12]:
def build_cross_db_summary(df):
    rows = []
    for (rsid, gene), group in df.groupby(["rsID", "gene"]):
        datasets = sorted(group["source_dataset"].unique())
        row = {
            "rsID":       rsid,
            "gene":       gene,
            "n_datasets": len(datasets),
            "datasets":   "; ".join(datasets),
        }
        for ds in datasets:
            ds_rows = group[group["source_dataset"] == ds]
            best = ds_rows.loc[ds_rows["p_value"].idxmin()]
            row[f"p_value_{ds}"]     = best["p_value"]
            row[f"effect_size_{ds}"] = best["effect_size"] if pd.notna(best["effect_size"]) else best["odds_ratio"]
        rows.append(row)

    summary_df = pd.DataFrame(rows)
    summary_df = summary_df.sort_values(["n_datasets", "gene"], ascending=[False, True]).reset_index(drop=True)
    return summary_df

cross_db_df = build_cross_db_summary(hf_df)

print(f"Cross-database summary: {len(cross_db_df):,} unique rsID/gene pairs")
print(f"Replicated in more than one dataset: {(cross_db_df['n_datasets'] > 1).sum():,}")
cross_db_df.head(10)

Cross-database summary: 51,000 unique rsID/gene pairs
Replicated in more than one dataset: 19,559


,rsID,gene,n_datasets,datasets,p_value_FinnGen,effect_size_FinnGen,p_value_HERMES,effect_size_HERMES,p_value_GWAS Catalog,effect_size_GWAS Catalog
0,rs7412,APOE,3,FinnGen; GWAS Catalog; HERMES,6.039490e-03,0.048592,0.643400,-0.0081,4.000000e-12,NaN
1,rs10119,APOE,2,FinnGen; HERMES,1.426490e-06,-0.044561,0.003862,-0.0283,NaN,NaN
2,rs1038025,APOE,2,FinnGen; HERMES,1.051910e-03,0.026753,0.361600,0.0098,NaN,NaN
3,rs1038026,APOE,2,FinnGen; HERMES,1.057060e-03,0.026741,0.919500,-0.0009,NaN,NaN
4,rs10402642,APOE,2,FinnGen; HERMES,1.844250e-01,-0.010807,0.511200,0.0057,NaN,NaN
5,rs10413089,APOE,2,FinnGen; HERMES,8.792000e-01,0.001678,0.136300,-0.0171,NaN,NaN
6,rs10413096,APOE,2,FinnGen; HERMES,5.411280e-03,-0.023004,0.261800,0.0103,NaN,NaN
7,rs10414043,APOE,2,FinnGen; HERMES,3.371790e-07,-0.058382,0.000748,-0.0471,NaN,NaN
8,rs10419086,APOE,2,FinnGen; HERMES,1.789040e-02,-0.056619,0.510500,-0.0147,NaN,NaN
9,rs10420434,APOE,2,FinnGen; HERMES,1.788960e-02,-0.056615,0.704000,0.0085,NaN,NaN


In [13]:
with pd.ExcelWriter(out_hf_xlsx, engine="openpyxl") as writer:
    hf_df.to_excel(writer, sheet_name="HF Variants", index=False)
    cross_db_df.to_excel(writer, sheet_name="Cross-Database Summary", index=False)

hf_df.to_csv(out_hf_csv, index=False, sep=";")